In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error



In [2]:
# Define paths (adjust if running in a different environment)
BASE_PATH = "/kaggle/input/petfinder-pawpularity-score"
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)



In [3]:
# Feature columns (metadata only)
FEATURE_COLS = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]

X = train_df[FEATURE_COLS].values.astype(np.float32)
y = train_df["Pawpularity"].values.astype(np.float32)



In [4]:
# Quick validation to see the current RMSE
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

gbr = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42
)
gbr.fit(X_train, y_train)
val_pred = gbr.predict(X_val)
val_rmse = mean_squared_error(y_val, val_pred, squared=False)
print(f"Validation RMSE: {val_rmse:.5f}")



Validation RMSE: 21.43535


In [5]:
# Retrain on the full training data
gbr_full = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42
)
gbr_full.fit(X, y)



GradientBoostingRegressor(learning_rate=0.05, n_estimators=500, random_state=42)

In [6]:
# Predict on test set
X_test = test_df[FEATURE_COLS].values.astype(np.float32)
test_pred = gbr_full.predict(X_test)

# Clip predictions to the valid range [0, 100] and round to two decimals
test_pred = np.clip(test_pred, 0, 100)
test_pred = np.round(test_pred, 2)



In [7]:
# Build submission DataFrame with required column names
submission = pd.DataFrame({"Id": test_df["Id"], "Pawpularity": test_pred})

# Write to CSV (Kaggle expects exactly this filename)
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
